In [67]:
import torch
import joblib
from torch import nn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error

In [51]:
# Upload window
from google.colab import files
uploaded = files.upload()

# Load the dataset
df = pd.read_excel(list(uploaded.keys())[0])
df.head()

Saving TrainDataset2025.xls to TrainDataset2025 (1).xls


,ID,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002174,1,144.0,41.0,0,0,0,1,3,3,...,0.517172,0.375126,3.325332,0.002314,3880771.500,473.464852,0.000768,0.182615,0.030508,0.000758
1,TRG002178,0,142.0,39.0,1,1,0,0,3,3,...,0.444391,0.444391,3.032144,0.005612,2372009.744,59.459710,0.004383,0.032012,0.001006,0.003685
2,TRG002204,1,135.0,31.0,0,0,0,1,2,1,...,0.534549,0.534549,2.485848,0.006752,1540027.421,33.935384,0.007584,0.024062,0.000529,0.006447
3,TRG002206,0,12.0,35.0,0,0,0,1,3,3,...,0.506185,0.506185,2.606255,0.003755,6936740.794,46.859265,0.005424,0.013707,0.000178,0.004543
4,TRG002210,0,109.0,61.0,1,0,0,0,2,1,...,0.462282,0.462282,2.809279,0.006521,1265399.054,39.621023,0.006585,0.034148,0.001083,0.005626


In [52]:
print("Shape: ", df.shape)

Shape:  (400, 121)


In [53]:
print("First 15 columns: ")
print(df.columns[:15])

First 15 columns: 
Index(['ID', 'pCR (outcome)', 'RelapseFreeSurvival (outcome)', 'Age', 'ER',
       'PgR', 'HER2', 'TrippleNegative', 'ChemoGrade', 'Proliferation',
       'HistologyType', 'LNStatus', 'TumourStage', 'Gene',
       'original_shape_Elongation'],
      dtype='object')


In [54]:
print("Summary of RFS outcome: ")
print(df["RelapseFreeSurvival (outcome)"].describe())

Summary of RFS outcome: 
count    400.000000
mean      56.000208
std       27.137584
min        0.000000
25%       38.000000
50%       55.000000
75%       73.000000
max      144.000000
Name: RelapseFreeSurvival (outcome), dtype: float64


In [55]:
missing_val = (df == 999). sum().sort_values(ascending=False)
print("Missing values: ")
print(missing_val)

Missing values: 
Gene                         88
pCR (outcome)                 5
ChemoGrade                    3
HistologyType                 3
Proliferation                 2
                             ..
original_ngtdm_Busyness       0
original_ngtdm_Coarseness     0
original_ngtdm_Complexity     0
original_ngtdm_Contrast       0
original_ngtdm_Strength       0
Length: 121, dtype: int64


In [56]:
df_RFS = df.copy()
df_RFS = df_RFS.replace(999, np.nan)
df_RFS.isna().sum().sort_values(ascending=False).head(10)

,0
Gene,88
pCR (outcome),5
ChemoGrade,3
HistologyType,3
Proliferation,2
LNStatus,1
TrippleNegative,1
PgR,1
HER2,1
Age,0


In [57]:
df_RFS = df_RFS.drop(columns=["ID"])

y_rfs = df_RFS["RelapseFreeSurvival (outcome)"]

X = df_RFS.drop(columns=["RelapseFreeSurvival (outcome)", "pCR (outcome)"])

print("Feature shape: ", X.shape)
print("Feature columns: ", X.columns)

Feature shape:  (400, 118)
Feature columns:  Index(['Age', 'ER', 'PgR', 'HER2', 'TrippleNegative', 'ChemoGrade',
       'Proliferation', 'HistologyType', 'LNStatus', 'TumourStage',
       ...
       'original_glszm_SmallAreaHighGrayLevelEmphasis',
       'original_glszm_SmallAreaLowGrayLevelEmphasis',
       'original_glszm_ZoneEntropy', 'original_glszm_ZonePercentage',
       'original_glszm_ZoneVariance', 'original_ngtdm_Busyness',
       'original_ngtdm_Coarseness', 'original_ngtdm_Complexity',
       'original_ngtdm_Contrast', 'original_ngtdm_Strength'],
      dtype='object', length=118)


In [58]:
X.dtypes.head(20)

,0
Age,float64
ER,int64
PgR,float64
HER2,float64
TrippleNegative,float64
ChemoGrade,float64
Proliferation,float64
HistologyType,float64
LNStatus,float64
TumourStage,int64


In [75]:
X_np = X.values.astype(np.float32)
y_np = y_rfs.values.astype(np.float32).reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)


imputer = IterativeImputer(
    random_state = 42,
    max_iter = 10,
    initial_strategy = "median"
)

X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)

im_path = "/content/drive/MyDrive/colab_2025/COMP4139/Assignment2/iterative_imputer.pkl"
joblib.dump(imputer, im_path)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

sc_path = "/content/drive/MyDrive/colab_2025/COMP4139/Assignment2/scaler.pkl"
joblib.dump(scaler, sc_path)

X_train = torch.from_numpy(X_train.astype(np.float32))
X_val = torch.from_numpy(X_val.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_val = torch.from_numpy(y_val.astype(np.float32))

print(X_train.shape, X_val.shape)
print(y_train.shape, y_val.shape)


torch.Size([320, 118]) torch.Size([80, 118])
torch.Size([320, 1]) torch.Size([80, 1])


In [69]:
class RFSNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

In [70]:
model = RFSNet(X_train.shape[1])

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

n_epochs = 200
batch_size = 32

for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(X_train.size(0))

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i: i+batch_size]
        X_batch = X_train[idx]
        y_batch = y_train[idx]

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val)
            val_preds = val_preds.detach().cpu().numpy().flatten()
            val_true = y_val.detach().cpu().numpy().flatten()
            mae = mean_absolute_error(val_true, val_preds)
            print(f"Epoch {epoch+1}, train loss: {loss.item(): .4f}, Validation MAE: {mae:.4f}")

model_path = "/content/drive/MyDrive/colab_2025/COMP4139/Assignment2/rfs_model.pth"
torch.save(model.state_dict(), model_path)

Epoch 20, train loss:  4340.6362, Validation MAE: 57.1303
Epoch 40, train loss:  3421.1536, Validation MAE: 54.1563
Epoch 60, train loss:  3052.3123, Validation MAE: 48.8773
Epoch 80, train loss:  2132.1521, Validation MAE: 41.7704
Epoch 100, train loss:  1967.9591, Validation MAE: 33.9711
Epoch 120, train loss:  1326.8728, Validation MAE: 28.8554
Epoch 140, train loss:  803.2830, Validation MAE: 26.4881
Epoch 160, train loss:  836.8818, Validation MAE: 24.8485
Epoch 180, train loss:  450.9305, Validation MAE: 23.7547
Epoch 200, train loss:  560.6528, Validation MAE: 23.2553


In [64]:
# Upload window
from google.colab import files
uploaded = files.upload()

# Load the dataset
df_test = pd.read_excel(list(uploaded.keys())[0])
df_test.head()

Saving TestDatasetExample.xls to TestDatasetExample.xls


,ID,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002728,56.881588,0,0,0,1,3,3,999,0,...,0.194591,0.194591,2.846439,0.001281,4.168474e+06,131.044541,0.002335,0.109755,0.013383,0.002051
1,TRG002649,60.000000,0,0,1,0,2,1,1,0,...,0.309999,0.309996,2.975317,0.007253,1.736585e+05,23.967478,0.011285,0.055890,0.003163,0.009553
2,TRG002628,58.234086,0,0,0,1,3,3,1,1,...,0.328377,0.328377,3.785966,0.003185,3.607821e+06,223.279556,0.001334,0.101628,0.010844,0.001194


In [66]:
df_test = df_test.replace(999, np.nan)
test_ids = df_test["ID"].copy()

X_test = df_test.drop(columns=["ID"])
print("Feature shape: ", X_test.shape)
print("Feature columns: ", X_test.columns)

Feature shape:  (3, 118)
Feature columns:  Index(['Age', 'ER', 'PgR', 'HER2', 'TrippleNegative', 'ChemoGrade',
       'Proliferation', 'HistologyType', 'LNStatus', 'TumourStage',
       ...
       'original_glszm_SmallAreaHighGrayLevelEmphasis',
       'original_glszm_SmallAreaLowGrayLevelEmphasis',
       'original_glszm_ZoneEntropy', 'original_glszm_ZonePercentage',
       'original_glszm_ZoneVariance', 'original_ngtdm_Busyness',
       'original_ngtdm_Coarseness', 'original_ngtdm_Complexity',
       'original_ngtdm_Contrast', 'original_ngtdm_Strength'],
      dtype='object', length=118)


In [76]:
imputer = joblib.load(im_path)
scaler  = joblib.load(sc_path)

X_test_imputed = imputer.transform(X_test)
X_test_imputed = scaler.transform(X_test_imputed)

X_test_tensor = torch.tensor(X_test_imputed, dtype=torch.float32)

model = RFSNet(X_test_tensor.shape[1])
model.load_state_dict(torch.load(model_path))
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor)
    test_preds = test_preds.detach().cpu().numpy().flatten()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but IterativeImputer was fitted without feature names
  warnings.warn(


In [77]:
output = pd.DataFrame({
    "ID": test_ids,
    "RelapseFreeSurvival (outcome)": test_preds
})

output_filename = "/content/drive/MyDrive/colab_2025/COMP4139/Assignment2/RFSPrediction.csv"
output.to_csv(output_filename, index=False)

In [79]:
df_outcome = pd.read_csv("/content/drive/MyDrive/colab_2025/COMP4139/Assignment2/RFSPrediction.csv")
df_outcome.head()

,ID,RelapseFreeSurvival (outcome)
0,TRG002728,45.251892
1,TRG002649,30.016170
2,TRG002628,38.268875
